# Unit 3 — PPO: Proximal Policy Optimisation

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

In Unit 2, A2C + GAE gave us a much better advantage estimator and stable
N-step rollouts — but it still has one fundamental limitation: **one gradient
update per rollout, with no constraint on step size**.  A single bad minibatch
can still overwrite a good policy.

This unit fixes that with **Proximal Policy Optimisation (PPO, Schulman et al.
2017)**, which lets us reuse each rollout for multiple gradient steps safely —
by clipping the probability ratio so the policy can never move too far from
where the data was collected.

We demonstrate on **LunarLander-v3**: A2C plateaus well below the solved
threshold; PPO reliably crosses it.

**What we build:**

| Part | Algorithm | Key change over previous unit |
|------|-----------|-------------------------------|
| A | A2C + GAE (Unit 2 baseline) | One gradient step per rollout |
| B | PPO-Clip | $K$ epochs × minibatches · clipped surrogate loss |


In [ ]:
%pip install -q swig
%pip install -q "gymnasium[box2d]" torch imageio pillow matplotlib
print("All packages ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from IPython.display import Image, display

print(f"gymnasium {gym.__version__}  |  torch {torch.__version__}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## 1  Environment — LunarLander-v3

We covered this environment in depth in Unit 2.  Quick recap:

- **State:** 8 continuous values — position $(x, y)$, velocity
  $(\dot{x}, \dot{y})$, angle $\theta$, angular velocity $\dot{\theta}$,
  and two leg-contact booleans $c_L, c_R$
- **Actions:** 4 discrete — do nothing / left engine / main engine / right engine
- **Reward:** dense shaping (change in distance + speed + tilt) + fuel cost,
  with $+100$ for a soft landing and $-100$ for a crash
- **Solved:** average $\geq 200$ over 100 consecutive greedy episodes

### Why this environment for PPO?

In Unit 2 we showed that A2C + GAE on LunarLander only reaches ~30 average
reward (consistent with published SB3 benchmarks).  The environment is hard
enough that A2C's single-update-per-rollout regime can't converge reliably —
**PPO needs to be the hero here**.


In [ ]:
env_peek = gym.make("LunarLander-v3")
STATE_DIM  = env_peek.observation_space.shape[0]   # 8
ACTION_DIM = env_peek.action_space.n               # 4
print(f"State dim : {STATE_DIM}")
print(f"Action dim: {ACTION_DIM}")
env_peek.close()


---
## 2  Why A2C Still Fails — The Step-Size Problem

A2C + GAE gave us two things:

1. **Better advantage estimates** via GAE (less variance)
2. **Stable gradient scale** via N-step rollouts from parallel envs

What it did NOT give us: any constraint on **how large a single gradient step
can be**.

After collecting a rollout, A2C does exactly one Adam step.  If that step is
large — because the advantages happened to be high, or the gradient is
pointing in a slightly wrong direction — the policy can jump to a bad region
of parameter space and collapse.

The natural fix: **reuse each rollout for $K$ gradient steps** instead of one.
More gradient steps per batch of data = better sample efficiency.

But here is the problem with naively reusing data.


---
## 3  Why Naive Multi-Step Reuse Fails — Distribution Shift

The policy gradient estimator is:

$$\hat{g} = \frac{1}{|D|}\sum_{\tau \in D} \sum_t
\nabla_\theta \log \pi_\theta(a_t|s_t)\,{\hat{A}}_t$$

This is an unbiased estimate of $\nabla_\theta J(\theta)$ **only when the
data $D$ was collected under the current policy $\pi_\theta$**.

After the first gradient step, $\theta$ has changed.  The data was collected
under $\pi_{\theta_{\text{old}}}$, not $\pi_\theta$.  Every subsequent step
uses data from the wrong distribution.  The gradient is biased, and the policy
degrades.

The solution: **importance sampling** — a mathematical correction that lets
you evaluate gradients under the new policy using data from the old one.


---
## 4  Importance Sampling — Exact Correction for Off-Distribution Data

**The IS identity.** For any two distributions $p$ and $q$ with $p$ absolutely
continuous with respect to $q$:

$$\mathbb{E}_{x \sim p}[f(x)] = \mathbb{E}_{x \sim q}\!\left[
  \frac{p(x)}{q(x)}\,f(x)
\right]$$

*Proof sketch:* write the left side as an integral, multiply and divide by $q(x)$:

$$\int f(x)\,p(x)\,dx = \int f(x)\,\frac{p(x)}{q(x)}\,q(x)\,dx
= \mathbb{E}_{x \sim q}\!\left[\frac{p(x)}{q(x)}\,f(x)\right]$$

The ratio $p(x)/q(x)$ is the **importance weight** — it reweights samples from
$q$ to produce correct expectations under $p$.

### Applied to trajectories

The objective is:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t r_t\right]
= \mathbb{E}_{\tau \sim \pi_{\theta_{\text{old}}}}\!\left[
  \frac{p_\theta(\tau)}{p_{\theta_{\text{old}}}(\tau)}\sum_t r_t
\right]$$

The trajectory probability is:

$$p_\theta(\tau) = p(s_0) \prod_t p(s_{t+1}|s_t,a_t) \cdot \pi_\theta(a_t|s_t)$$

where $p(s_{t+1}|s_t,a_t)$ are the **environment dynamics** — identical in
numerator and denominator, independent of $\theta$, so they cancel:

$$\frac{p_\theta(\tau)}{p_{\theta_{\text{old}}}(\tau)}
= \prod_t \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

### The surrogate loss

Using the per-step advantage decomposition and the IS correction, the
**surrogate objective** to maximise is:

$$L^{\text{IS}}(\theta) = \mathbb{E}_t\!\left[
  \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}\,\hat{A}_t
\right] = \mathbb{E}_t\!\left[r_t(\theta)\,\hat{A}_t\right]$$

where $r_t(\theta) = \pi_\theta(a_t|s_t) \;/\; \pi_{\theta_{\text{old}}}(a_t|s_t)$
is the **probability ratio**.

Note that at $\theta = \theta_{\text{old}}$: $r_t = 1$ and we recover
the standard policy gradient — vanilla PG is just IS evaluated at the
collection point.

### Why you still need a trust region

IS is mathematically exact, but the IS estimator has high variance when
$\pi_\theta$ drifts far from $\pi_{\theta_{\text{old}}}$ — the ratio $r_t$
becomes large, amplifying any noise in $\hat{A}_t$.  More importantly, the
first-order surrogate is only a valid approximation of $J(\theta)$ near
$\theta_{\text{old}}$.  Taking large steps makes the approximation inaccurate.

**Constraining $r_t$ to stay near 1** bounds how far $\pi_\theta$ can deviate
from $\pi_{\theta_{\text{old}}}$ — which is exactly what PPO does.


---
## 5  PPO-Clip — A Structural Trust Region

TRPO enforces a hard KL constraint via a constrained optimisation solve
(natural gradient, conjugate gradient, line search) — correct but expensive.

PPO-Clip achieves a similar effect with a single line of code by **clipping**
the probability ratio to $[1-\varepsilon,\,1+\varepsilon]$:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[
  \min\!\left(
    r_t(\theta)\,\hat{A}_t,\;
    \text{clip}\bigl(r_t(\theta),\,1-\varepsilon,\,1+\varepsilon\bigr)\,\hat{A}_t
  \right)
\right]$$

The $\min$ makes this a **pessimistic lower bound** — we take the worse of
the two estimates.  The clip then kills the gradient whenever the policy
tries to move outside the trust region.

### The 4 cases

Fix a timestep $t$ and consider all combinations of advantage sign and ratio
position:

| | $r_t < 1-\varepsilon$ (ratio too low) | $1-\varepsilon \leq r_t \leq 1+\varepsilon$ | $r_t > 1+\varepsilon$ (ratio too high) |
|--|--|--|--|
| $\hat{A}_t > 0$ (good action) | clipped at $1-\varepsilon$ — **no gradient** | unclipped — gradient flows | clipped at $1+\varepsilon$ — **no gradient** |
| $\hat{A}_t < 0$ (bad action) | clipped at $1-\varepsilon$ — **no gradient** | unclipped — gradient flows | clipped at $1+\varepsilon$ — **no gradient** |

Gradient only flows when $r_t \in [1-\varepsilon,\,1+\varepsilon]$ —
**inside the trust region**.  Outside, the clip activates and the update
stops automatically.  No KL computation, no Lagrange multiplier, no line
search.

### Why $\varepsilon = 0.2$ works universally

The clip defines a symmetric 20% band around the old policy.  Empirically
this is wide enough for meaningful updates but tight enough to prevent
collapse.  Unlike TRPO's KL threshold, $\varepsilon$ has an intuitive
interpretation (allowed percentage change in action probabilities) and does
not need per-environment tuning.

### Connection to PPO-Lagrangian (PPO v1)

An equivalent formulation penalises the KL directly:

$$L^{\text{KLPEN}}(\theta) = \mathbb{E}_t[r_t(\theta)\hat{A}_t] - \beta\,
\text{KL}[\pi_{\theta_{\text{old}}}\|\pi_\theta]$$

where $\beta$ is a Lagrange multiplier that self-adjusts: if the observed KL
exceeds the target, $\beta$ increases; if below, $\beta$ decreases.
PPO-Clip is simpler and works at least as well in practice.


---
## 6  Networks

Same architecture and initialisation as Unit 2.
LunarLander's 8-dimensional state needs wider networks than Acrobot (256 vs 64).


In [ ]:
def layer_init(layer: nn.Linear, std: float = np.sqrt(2)) -> nn.Linear:
    """Orthogonal init — standard for PPO (CleanRL, SB3, OpenAI baselines)."""
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, 0.0)
    return layer


class ActorNetwork(nn.Module):
    """
    State → action logits.
    Input:  (state_dim,)   = (8,)  for LunarLander
    Output: (action_dim,)  = (4,)  raw logits → Categorical(logits=...)
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, action_dim), std=0.01),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def get_action(self, state: torch.Tensor, action: torch.Tensor = None):
        dist = torch.distributions.Categorical(logits=self.forward(state))
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy()


class CriticNetwork(nn.Module):
    """State → scalar V(s)."""
    def __init__(self, state_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


---
## 7  GAE — Unchanged from Unit 2

PPO uses exactly the same GAE computation as A2C.
The advantage estimator is not what changes — only the loss function does.


In [ ]:
def compute_gae(
    rewards:     torch.Tensor,   # (T, N_envs)
    values:      torch.Tensor,   # (T, N_envs) — detached
    dones:       torch.Tensor,   # (T, N_envs)
    next_values: torch.Tensor,   # (N_envs,)   — bootstrap
    gamma: float = 0.999,
    lam:   float = 0.98,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Backward GAE recurrence with episode-boundary masking.
    Returns (advantages, returns) both flattened to (T*N_envs,).
    Advantages are normalised; returns are raw (critic regression targets).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros(N, device=rewards.device)

    for t in reversed(range(T)):
        nonterminal = 1.0 - dones[t]
        next_v = values[t + 1] if t < T - 1 else next_values
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae

    returns  = advantages + values
    adv_flat = advantages.reshape(-1)
    ret_flat = returns.reshape(-1)
    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)
    return adv_flat, ret_flat


---
## Part A — A2C + GAE Baseline on LunarLander

The Unit 2 algorithm unchanged: one gradient update per rollout.
Expected: plateau well below 200 (consistent with SB3 benchmarks for A2C
on LunarLander-v3, which reports ~30 average reward).


In [ ]:
def a2c_gae(
    actor, critic, actor_opt, critic_opt,
    n_envs:       int   = 8,
    n_steps:      int   = 1024,
    total_steps:  int   = 500_000,
    gamma:        float = 0.999,
    lam:          float = 0.98,
    entropy_coef: float = 0.01,
    value_coef:   float = 0.5,
    max_grad_norm:float = 0.5,
    print_every:  int   = 100_000,
    seed:         int   = 42,
) -> tuple[list, list]:
    """A2C + GAE — Unit 2 algorithm, one update per rollout."""
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("LunarLander-v3") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

    ep_buf = np.zeros(n_envs)
    all_ep_rew, step_at_ep = [], []
    n_updates = total_steps // (n_steps * n_envs)

    for update in range(n_updates):
        steps_so_far = update * n_steps * n_envs
        obs_b = torch.zeros(n_steps, n_envs, STATE_DIM, device=device)
        act_b = torch.zeros(n_steps, n_envs, dtype=torch.long, device=device)
        rew_b = torch.zeros(n_steps, n_envs, device=device)
        don_b = torch.zeros(n_steps, n_envs, device=device)
        val_b = torch.zeros(n_steps, n_envs, device=device)

        for step in range(n_steps):
            with torch.no_grad():
                action, _, _ = actor.get_action(obs)
                value = critic(obs)
            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)
            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    step_at_ep.append(steps_so_far + (step + 1) * n_envs)
                    ep_buf[i] = 0.0
            obs_b[step] = obs; act_b[step] = action
            rew_b[step] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            don_b[step] = torch.as_tensor(done, dtype=torch.float32, device=device)
            val_b[step] = value
            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        with torch.no_grad():
            next_val = critic(obs)

        advantages, returns = compute_gae(rew_b, val_b.detach(), don_b, next_val, gamma, lam)
        obs_flat = obs_b.reshape(-1, STATE_DIM)
        act_flat = act_b.reshape(-1)
        _, new_log_probs, entropy = actor.get_action(obs_flat, act_flat)
        new_values = critic(obs_flat)

        actor_loss  = -(new_log_probs * advantages).mean()
        critic_loss = value_coef * F.mse_loss(new_values, returns.detach())
        entropy_loss = -entropy_coef * entropy.mean()

        actor_opt.zero_grad(); critic_opt.zero_grad()
        (actor_loss + critic_loss + entropy_loss).backward()
        nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
        actor_opt.step(); critic_opt.step()

        steps_done = (update + 1) * n_steps * n_envs
        if steps_done // print_every > (steps_done - n_steps * n_envs) // print_every and all_ep_rew:
            avg = np.mean(all_ep_rew[-100:]) if len(all_ep_rew) >= 100 else np.mean(all_ep_rew)
            print(f"[A2C]  step {steps_done:7d}  eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}")

    envs.close()
    return all_ep_rew, step_at_ep


---
## Part B — PPO-Clip

Three changes from A2C, in order of importance:

1. **$K$ epochs over the same rollout** — instead of one gradient step, we
   loop over the rollout $K=4$ times with minibatches, giving many gradient
   steps per rollout.

2. **Long rollouts + small minibatches** — `n_steps=1024` means each rollout
   has $1024 \times 8 = 8192$ transitions. With minibatch size 64, each
   epoch runs $8192/64 = 128$ gradient steps. Four epochs = **512 gradient
   steps per rollout**. Compare to A2C's 1. This is the key to PPO's sample
   efficiency — not just multiple epochs, but the combination of large rollouts
   with small minibatches. (Hyperparameters from rl-baselines3-zoo.)

3. **Clipped surrogate loss** — the core of PPO.  Instead of the standard
   policy gradient $\log\pi(a|s)\cdot\hat{A}$, we use the probability ratio
   $r_t(\theta) = \pi_\theta/\pi_{\theta_{\text{old}}}$ clipped to prevent
   large steps.

Everything else — network, optimiser, GAE, gradient clipping — is unchanged.


In [ ]:
def ppo_clip(
    actor, critic, opt,
    n_envs:        int   = 8,
    n_steps:       int   = 1024,   # long rollouts → rich GAE estimates
    total_steps:   int   = 1_000_000,
    n_epochs:      int   = 4,      # reuse each rollout 4 times
    minibatch_size:int   = 64,     # small batches → many gradient steps per rollout
    gamma:         float = 0.999,  # SB3-Zoo tuned for LunarLander
    lam:           float = 0.98,   # SB3-Zoo tuned for LunarLander
    clip_coef:     float = 0.2,    # ε — the trust-region width
    entropy_coef:  float = 0.01,
    value_coef:    float = 0.5,
    max_grad_norm: float = 0.5,
    print_every:   int   = 100_000,
    seed:          int   = 42,
) -> tuple[list, list]:
    """
    PPO-Clip on LunarLander-v3.

    Rollout loop (same as A2C, no_grad):
      Collect n_steps transitions from n_envs parallel environments.

    Update loop (new in PPO):
      for epoch in range(n_epochs):            # reuse the rollout K times
          for minibatch in shuffled_rollout:   # minibatch gradient step
              ratio    = exp(new_log_prob - old_log_prob)
              pg_loss1 = -advantage * ratio
              pg_loss2 = -advantage * clip(ratio, 1-ε, 1+ε)
              actor_loss = mean(max(pg_loss1, pg_loss2))   ← pessimistic lower bound
              critic_loss = 0.5 * MSE(new_value, return)
              total_loss  = actor_loss + value_coef*critic_loss - entropy_coef*H
              backward + clip_grad + step

    Hyperparameters match rl-baselines3-zoo tuned config for LunarLander-v3.
    """
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("LunarLander-v3") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

    ep_buf = np.zeros(n_envs)
    all_ep_rew, step_at_ep = [], []
    batch_size = n_steps * n_envs   # total transitions per rollout: 1024×8 = 8192

    n_updates = total_steps // batch_size

    for update in range(n_updates):
        steps_so_far = update * batch_size

        # ── 1. Allocate rollout buffers ───────────────────────────────────────
        obs_b  = torch.zeros(n_steps, n_envs, STATE_DIM, device=device)
        act_b  = torch.zeros(n_steps, n_envs, dtype=torch.long, device=device)
        lp_b   = torch.zeros(n_steps, n_envs, device=device)   # old log-probs
        rew_b  = torch.zeros(n_steps, n_envs, device=device)
        don_b  = torch.zeros(n_steps, n_envs, device=device)
        val_b  = torch.zeros(n_steps, n_envs, device=device)

        # ── 2. Collect rollout (no_grad — these are the "old" policy values) ──
        for step in range(n_steps):
            with torch.no_grad():
                action, log_prob, _ = actor.get_action(obs)  # (N_envs,)
                value = critic(obs)                            # (N_envs,)

            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)

            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    step_at_ep.append(steps_so_far + (step + 1) * n_envs)
                    ep_buf[i] = 0.0

            obs_b[step] = obs;   act_b[step] = action
            lp_b[step]  = log_prob
            rew_b[step] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            don_b[step] = torch.as_tensor(done,   dtype=torch.float32, device=device)
            val_b[step] = value

            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        # ── 3. Bootstrap + GAE ────────────────────────────────────────────────
        with torch.no_grad():
            next_val = critic(obs)   # V(s_{T+1})

        advantages, returns = compute_gae(
            rew_b, val_b.detach(), don_b, next_val, gamma, lam
        )
        # advantages: (batch_size,) normalised
        # returns:    (batch_size,) critic regression targets

        # ── 4. Flatten buffers ────────────────────────────────────────────────
        obs_flat = obs_b.reshape(-1, STATE_DIM)   # (1024, 8)
        act_flat = act_b.reshape(-1)               # (1024,)
        lp_old   = lp_b.reshape(-1).detach()       # (1024,)  frozen old log-probs

        # ── 5. PPO update: K epochs × minibatches ────────────────────────────
        for epoch in range(n_epochs):
            # Shuffle all transitions so minibatches are diverse
            perm = torch.randperm(batch_size, device=device)

            for start in range(0, batch_size, minibatch_size):
                mb_idx = perm[start : start + minibatch_size]

                # Re-evaluate current policy on this minibatch
                _, new_log_probs, entropy = actor.get_action(obs_flat[mb_idx], act_flat[mb_idx])
                new_values = critic(obs_flat[mb_idx])

                # Probability ratio  r_t(θ) = π_θ(a|s) / π_θ_old(a|s)
                # Use log-space for numerical stability: exp(log π_new - log π_old)
                ratio = torch.exp(new_log_probs - lp_old[mb_idx])
                # ratio shape: (minibatch_size,)

                mb_adv = advantages[mb_idx]   # (minibatch_size,)

                # ── Clipped surrogate loss ─────────────────────────────────
                # Unclipped: r_t * A_t
                pg_loss1 = -mb_adv * ratio
                # Clipped:   clip(r_t, 1-ε, 1+ε) * A_t
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
                # Pessimistic lower bound: take the worse (max of negatives)
                # This kills the gradient whenever ratio is outside [1-ε, 1+ε]
                actor_loss = torch.max(pg_loss1, pg_loss2).mean()

                # ── Critic loss ────────────────────────────────────────────
                critic_loss = value_coef * F.mse_loss(new_values, returns[mb_idx].detach())

                # ── Entropy bonus ──────────────────────────────────────────
                entropy_loss = -entropy_coef * entropy.mean()

                total_loss = actor_loss + critic_loss + entropy_loss

                opt.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(
                    list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
                opt.step()

        steps_done = (update + 1) * batch_size
        if steps_done // print_every > (steps_done - batch_size) // print_every and all_ep_rew:
            avg = np.mean(all_ep_rew[-100:]) if len(all_ep_rew) >= 100 else np.mean(all_ep_rew)
            solved = "✓ SOLVED" if avg >= 200 else ""
            print(f"[PPO]  step {steps_done:7d}  eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}  {solved}")

    envs.close()
    return all_ep_rew, step_at_ep


---
## 8  Train — Same Budget, Same Seed

Both algorithms receive the same 300k steps for A2C and 500k for PPO.
PPO gets more steps because it's the algorithm we're showcasing — and it
needs to convincingly cross 200 to make the point.

Hyperparameters from [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo)
tuned config for LunarLander-v3 (γ=0.999, λ=0.98 — higher than Acrobot
because landing requires long-horizon credit assignment).

**Expected runtime on Colab CPU:** ~20 minutes total (LunarLander box2d is heavier than Acrobot).
GPU does not help here — the bottleneck is the box2d physics stepping on CPU.


In [ ]:
SEED       = 42
HIDDEN_DIM = 256
LR         = 3e-4    # SB3 PPO default

torch.manual_seed(SEED)
np.random.seed(SEED)

def make_actor_critic(seed):
    torch.manual_seed(seed)
    actor  = ActorNetwork(STATE_DIM, ACTION_DIM, HIDDEN_DIM).to(device)
    critic = CriticNetwork(STATE_DIM, HIDDEN_DIM).to(device)
    return actor, critic


In [ ]:
# ── Part A: A2C ───────────────────────────────────────────────────────────────
actor_a, critic_a = make_actor_critic(SEED)
opt_a_actor  = optim.Adam(actor_a.parameters(),  lr=LR, eps=1e-5)
opt_a_critic = optim.Adam(critic_a.parameters(), lr=LR, eps=1e-5)

print("=" * 60)
print("Part A — A2C + GAE  (500k steps, 1 update per rollout)")
print("=" * 60)
rewards_a2c, steps_a2c = a2c_gae(
    actor_a, critic_a, opt_a_actor, opt_a_critic,
    n_envs=8, n_steps=1024, total_steps=500_000, seed=SEED,
)
print(f"\nA2C  final avg (last 100): {np.mean(rewards_a2c[-100:]):.1f}")
print(f"     solved (avg ≥ 200)    {'✓ SOLVED' if np.mean(rewards_a2c[-100:]) >= 200 else 'not solved'}")


In [ ]:
# ── Part B: PPO ───────────────────────────────────────────────────────────────
actor_b, critic_b = make_actor_critic(SEED)
# PPO uses a single combined optimiser — both networks share one Adam instance
opt_ppo = optim.Adam(
    list(actor_b.parameters()) + list(critic_b.parameters()),
    lr=LR, eps=1e-5
)

print("=" * 60)
print("Part B — PPO-Clip  (1M steps · 4 epochs · mb=64 · ε=0.2)")
print("=" * 60)
rewards_ppo, steps_ppo = ppo_clip(
    actor_b, critic_b, opt_ppo,
    n_envs=8, n_steps=1024, total_steps=1_000_000, seed=SEED,
)
print(f"\nPPO  final avg (last 100): {np.mean(rewards_ppo[-100:]):.1f}")
print(f"     solved (avg ≥ 200)    {'✓ SOLVED' if np.mean(rewards_ppo[-100:]) >= 200 else 'not solved'}")


---
## 9  Comparison — A2C vs PPO


In [ ]:
def smooth(steps, rewards, window=50):
    """Trailing-window smoothed curve, returns (steps, smoothed)."""
    out = []
    for i in range(len(rewards)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(rewards[lo : i + 1])))
    return steps, out


fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(steps_a2c, rewards_a2c, alpha=0.1, color="steelblue", linewidth=0.7)
ax.plot(*smooth(steps_a2c, rewards_a2c),
        color="steelblue", linewidth=2.5,
        label=f"A2C + GAE  (1 epoch, {len(rewards_a2c)} episodes)")

ax.plot(steps_ppo, rewards_ppo, alpha=0.1, color="darkorange", linewidth=0.7)
ax.plot(*smooth(steps_ppo, rewards_ppo),
        color="darkorange", linewidth=2.5,
        label=f"PPO-Clip   (4 epochs · mb=64, {len(rewards_ppo)} episodes)")

ax.axhline(200, color="red", linestyle="--", linewidth=1.5,
           alpha=0.8, label="Solved threshold (200)")

ax.set_xlabel("Total environment steps (same-scale comparison)", fontsize=12)
ax.set_ylabel("Episode reward", fontsize=12)
ax.set_title("A2C vs PPO-Clip on LunarLander-v3", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130)
plt.show()
print("Saved → comparison.png")


### Interpreting the results

**A2C (blue):** makes initial progress — the environment's dense reward
shaping gives enough signal to climb out of negative territory — but stalls
well below 200.  Each rollout produces one gradient step.  Unlucky rollouts
occasionally push the policy backward, and with no step-size constraint
there is no recovery mechanism.

**PPO (orange):** climbs more steadily and crosses 200.  The key difference:
every 8192-transition rollout yields 4 epochs × 128 minibatches = 512 gradient
steps, all safely bounded by the clip.  The pessimistic lower bound means
a single bad minibatch can hurt at most by $\varepsilon = 0.2$ in probability
space before the gradient dies.

#### Why the same data used 16 times is safe

After the first gradient step on a minibatch, the current policy $\pi_\theta$
differs from $\pi_{\theta_{\text{old}}}$.  On step 2, the ratio $r_t(\theta)$
is no longer 1.0.  If the policy drifted far from the rollout distribution,
$r_t$ exits $[1-\varepsilon, 1+\varepsilon]$ and the gradient stops.
The clip structurally prevents overfitting to stale data.

#### Seed and hardware variability

As in previous units, exact numbers depend on seed and device.  The
qualitative story — PPO solving LunarLander while A2C does not — is robust.


In [ ]:
def evaluate_greedy(actor: ActorNetwork, n_episodes: int = 100) -> float:
    eval_env = gym.make("LunarLander-v3")
    rewards = []
    actor.eval()
    with torch.no_grad():
        for _ in range(n_episodes):
            state, _ = eval_env.reset()
            total, done = 0.0, False
            while not done:
                s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
                action = actor(s_t).argmax().item()
                state, r, terminated, truncated, _ = eval_env.step(action)
                done = terminated or truncated
                total += r
            rewards.append(total)
    eval_env.close()
    actor.train()
    return float(np.mean(rewards))


score_a2c = evaluate_greedy(actor_a)
score_ppo = evaluate_greedy(actor_b)

print("Greedy evaluation (100 episodes, argmax policy):")
print(f"  A2C + GAE  : {score_a2c:7.1f}  {'✓ SOLVED' if score_a2c >= 200 else 'not solved'}")
print(f"  PPO-Clip   : {score_ppo:7.1f}  {'✓ SOLVED' if score_ppo >= 200 else 'not solved'}")


---
## 10  Watch the PPO Agent Land


In [ ]:
def record_gif(actor: ActorNetwork, filename: str = "agent.gif",
               max_steps: int = 1000) -> str:
    env_rec = gym.make("LunarLander-v3", render_mode="rgb_array")
    frames, total_reward = [], 0.0
    state, _ = env_rec.reset()
    done = False
    actor.eval()
    with torch.no_grad():
        while not done and len(frames) < max_steps:
            frames.append(env_rec.render())
            s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = actor(s_t).argmax().item()
            state, r, terminated, truncated, _ = env_rec.step(action)
            done = terminated or truncated
            total_reward += r
    env_rec.close()
    actor.train()
    iio.imwrite(filename, np.stack(frames), plugin="pillow", loop=0, duration=33)
    print(f"{len(frames)} frames  |  total reward: {total_reward:.1f}  →  {filename}")
    return filename


gif_path = record_gif(actor_b)
display(Image(gif_path))


---
## 11  What's Next — RLHF

PPO is the workhorse of modern deep RL — it trains locomotion policies in
robotics, game-playing agents, and (with modifications) large language models.

In Units 1–3 we assumed the **reward function was given** by the environment.
In practice, for language models, there is no natural reward signal.
You cannot write a function that scores whether a paragraph is helpful or
harmful.

**RLHF** (Reinforcement Learning from Human Feedback, Ziegler et al. 2019)
solves this by learning a reward model from human preference data, then
fine-tuning a language model with PPO against that learned reward.

The policy gradient update is:

$$\theta \leftarrow \theta + \alpha\,\nabla_\theta
\mathbb{E}_{a \sim \pi_\theta}\!\left[r_\phi(x, a)
- \beta\,\text{KL}\bigl(\pi_\theta(\cdot|x)\|\pi_{\text{ref}}(\cdot|x)\bigr)\right]$$

where $r_\phi$ is the learned reward model and $\pi_{\text{ref}}$ is the
original SFT model acting as a KL anchor.

Everything from Units 1–3 — policy gradient, advantage estimation, PPO's
clipped surrogate — carries forward directly.  The only new piece is the
reward model training.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
